# 2 — One arm, one question, both tiers

The full sweep is 1,440 of these. This is one, slowly, with everything visible:
the tools the agent is handed, the instruction it gets, the calls it makes, and
how the answer is scored.

**This makes live model calls** — a couple of minutes and a few hundred thousand
tokens. It writes nothing to `results/`.

Needs [`01_provision.ipynb`](01_provision.ipynb) (or `make bootstrap`) to have run
first, and `make toolbox` for the self-hosted arms — `mcp_clients.bind` launches
the pinned binary per cell and shuts it down again.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path[:0] = [str(ROOT / "src")]

import agents
import battery
import config
import golden
import mcp_clients
import scoring
from google.cloud import bigquery

# Change these two and re-run. `mcp_clients.CONFIG_KEYS` lists every arm;
# `examples/questions.json` lists every question.
CONFIG_KEY = "p1_toolbox"
QUESTION_ID = "semantic-q1"

question = next(q for q in battery.load_questions() if q.id == QUESTION_ID)
arm = mcp_clients.CONFIGS[CONFIG_KEY]

print(f"arm       {arm.key}  (path {arm.path}, {arm.variant}) — {arm.name}")
print(f"          {arm.summary}")
print(f"question  [{question.category}] {question.question}")
print(f"oracle    {question.golden_key}")
print(f"evidence  {question.evidence}")

/home/user/git/vertex-ai-mlops/Applied ML/AI Agents/data-mcp-sandbox/.venv/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


arm       p1_toolbox  (path 1, toolbox) — Raw Data Builder
          Self-hosted BigQuery source. Same job, wider tool surface.
question  [semantic-ambiguity] What is our total net revenue, all time?
oracle    net_revenue_all_time
evidence  {'must_have': ['txn_amt_x2', 'status_flg'], 'nice_to_have': ['transactions_v2_final'], 'distractor': ['revenue_amount']}


## What the arm actually binds

Tool declarations are re-sent to the model on **every turn**, so their serialized
size sets a floor under the prompt tokens for the whole conversation. Worth
seeing for one arm before reading it for ten: this is the mechanism behind the
entire cost result, and it is measured off the live endpoint rather than assumed.

Try `p1_managed` and `p1_matched` here. They bind the **same five tools** and are
two orders of magnitude apart in schema size — see
[`docs/paths.md`](../docs/paths.md).

In [2]:
print(await battery.measure_schemas([CONFIG_KEY]))

mTLS was requested but AsyncAuthorizedSession channel is not mTLS


{'p1_toolbox': {'tools': 8, 'schema_chars': 7030}}


/home/user/git/vertex-ai-mlops/Applied ML/AI Agents/data-mcp-sandbox/.venv/lib/python3.11/site-packages/google/adk/tools/mcp_tool/mcp_toolset.py:400: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(
/home/user/git/vertex-ai-mlops/Applied ML/AI Agents/data-mcp-sandbox/.venv/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:71: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()
/home/user/git/vertex-ai-mlops/Applied ML/AI Agents/data-mcp-sandbox/src/battery.py:160: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  chars += len(str(tool._get_declaration()))  # noqa: SLF001


## The instruction

Identical across tiers apart from the dataset it points at. That is deliberate
and it is the whole design: **the tiers are separated by IAM and by what the
warehouse says about itself, never by the prompt.** Nothing below tells the agent
about net revenue, refunds, or what "active" means. If a tier-1 agent gets those
right, it read them off the governed warehouse.

In [3]:
print(agents.instruction(CONFIG_KEY, 1))

You are a data analyst answering questions about a company's data.

The data is in BigQuery, in the dataset `statmike-mlops-349915.data_mcp_sandbox_t1`, which contains
these tables:
  - `statmike-mlops-349915.data_mcp_sandbox_t1.users`
  - `statmike-mlops-349915.data_mcp_sandbox_t1.transactions_v2_final`
  - `statmike-mlops-349915.data_mcp_sandbox_t1.raw_events_2026`

Use only this dataset. Do not query any other dataset.

Work out the answer using the tools available to you. Inspect the data's
metadata before you query it, so that you use the columns that actually mean
what the question is asking about.

Finish with a short, direct answer. When the answer is a number, state that
number plainly, and state the SQL or query you used to get it. Do not hedge
between two candidate numbers — commit to one.



## Ask it, ungoverned then governed

Same model, same temperature, same question, same rows. Only the metadata
differs. `set_active_tier` keeps everything that reads `config.SCOPE` pointing at
the tier being run, exactly as the sweep does.

`agents.ask` returns errors inside the `Outcome` instead of raising, because a
cell that fails on its own merits is a data point about that architecture.

In [4]:
outcomes = {}
for tier in config.TIERS:
    config.set_active_tier(tier)
    got = outcomes[tier] = await agents.ask(CONFIG_KEY, tier, question.question)
    print(f"tier {tier}: {got.latency_s:>6.1f}s  {len(got.tool_calls)} tool calls  "
          f"{got.usage.get('total_tokens', 0):>8,} tokens "
          f"({got.usage.get('thought_tokens', 0):,} of them thinking)"
          + (f"\n        ERROR {got.error[:160]}" if got.error else ""))

mTLS was requested but AsyncAuthorizedSession channel is not mTLS


tier 0:   73.9s  17 tool calls   120,751 tokens (2,214 of them thinking)


mTLS was requested but AsyncAuthorizedSession channel is not mTLS


tier 1:   26.2s  7 tool calls    25,984 tokens (681 of them thinking)


## What it did

The capture stores every call with its arguments and its result, which is what
makes the scoring arguable rather than something you have to take on trust.
Results are truncated for storage; the arguments never are.

In [5]:
for tier, outcome in outcomes.items():
    print(f"===== tier {tier} — {config.TIER_LABELS[tier]} =====")
    for call in outcome.tool_calls:
        flag = "  FAILED" if call.is_error else ""
        print(f"  [{call.seq}] {call.name}  {call.duration_s:.1f}s{flag}")
        print(f"      args {call.args}")
        print(f"      ->   {call.result[:300]}")
    print(f"\n  ANSWER: {outcome.answer[:900]}\n")

===== tier 0 — 0 · ungoverned control =====
  [0] get_table_info  0.8s
      args {'project': 'statmike-mlops-349915', 'dataset': 'data_mcp_sandbox_t0', 'table': 'transactions_v2_final'}
      ->   {'content': [{'type': 'text', 'text': '{"Name":"","Location":"US","Description":"","Schema":[{"Name":"user_id","Description":"","Repeated":false,"Required":false,"Type":"STRING","PolicyTags":null,"Schema":null,"MaxLength":0,"Precision":0,"Scale":0,"DefaultValueExpression":"","Collation":"","RangeEle
  [1] get_table_info  0.8s
      args {'table': 'raw_events_2026', 'dataset': 'data_mcp_sandbox_t0', 'project': 'statmike-mlops-349915'}
      ->   {'content': [{'type': 'text', 'text': '{"Name":"","Location":"US","Description":"","Schema":[{"Name":"event_id","Description":"","Repeated":false,"Required":false,"Type":"STRING","PolicyTags":null,"Schema":null,"MaxLength":0,"Precision":0,"Scale":0,"DefaultValueExpression":"","Collation":"","RangeEl
  [2] get_table_info  0.8s
      args {'dataset': 'd

## Score it

Through the same two functions the sweep and the report use — `battery.to_cell`
then `scoring.score_cell` — so a cell run by hand here is identical in shape to
one the sweep wrote, and is judged by exactly the same rubric.

The oracle resolves against live BigQuery because the corpus anchors its
timestamps to build time, so a stored number rots. `make export` freezes the
oracle into a published capture; see [`03_results.ipynb`](03_results.ipynb).

In [6]:
client = bigquery.Client(project=config.require_project())

for tier, outcome in outcomes.items():
    cell = battery.to_cell(question, CONFIG_KEY, tier, 1, outcome)
    truth = golden.resolve(client, question.golden_key, tier)
    score = scoring.score_cell(cell, question.evidence, truth)

    print(f"===== tier {tier} =====")
    print(f"  extracted     {score.value}")
    print(f"  truth         {truth.value:,.2f}   (+/- {truth.tolerance:.1%})")
    print(f"  trap value    {truth.trap_value}   [{truth.trap_name}]")
    print(f"  correct       {score.correct}")
    print(f"  sprang trap   {score.sprang_trap}")
    print(f"  rules needed  {score.rules_required}")
    print(f"  rules seen    {score.rules_acquired}  (observable={score.acquisition_observable})")
    print(f"  evidence      recall={score.evidence_recall} "
          f"precision={score.evidence_precision} (observable={score.evidence_observable})")
    print(f"  application loss  {score.application_loss}")
    for note in score.notes:
        print(f"    - {note}")
    print()

===== tier 0 =====
  extracted     31700036.25
  truth         4,032,361.00   (+/- 0.5%)
  trap value    31700036.25   [summed gross list price instead of net revenue]
  correct       False
  sprang trap   True
  rules needed  ['net-revenue']
  rules seen    []  (observable=True)
  evidence      recall=1.0 precision=0.6666666666666666 (observable=True)
  application loss  False



===== tier 1 =====
  extracted     4032361.0
  truth         4,032,361.00   (+/- 0.5%)
  trap value    31700036.25   [summed gross list price instead of net revenue]
  correct       True
  sprang trap   False
  rules needed  ['net-revenue']
  rules seen    ['net-revenue']  (observable=True)
  evidence      recall=1.0 precision=0.6666666666666666 (observable=True)
  application loss  False



## What to look for

The interesting tier-0 failure is not "it could not answer". It answers
confidently and it is wrong: it finds `revenue_amount`, which is gross, and has
no way to know that. `sprang_trap` is the column that records the difference
between being wrong and being wrong *in the way the corpus set out to catch*.

The other one to watch for is a tier-1 cell where `rules_seen` is full and
`correct` is still false — `application_loss`. The governed rule reached the
agent and the agent got the answer wrong anyway. Across the whole sweep that is
30–40% of governed cells, and it is what the headline result is really about:
acquiring metadata is close to solved, applying it is not.

Note also which fields read `None` rather than `0.0`. On the Conversational
Analytics arms the intermediate SQL is not exposed, so evidence is *unmeasurable*
on that path — recording it as zero would report a missing instrument as a failed
agent.

## Next

```bash
make plan     # what a full sweep costs, in time and tokens. Free.
make smoke    # twelve cells, one per arm
make sweep    # the full 1,440
```

Then [`03_results.ipynb`](03_results.ipynb).